# LLM from Scratch — Lab

**Audience:** SEMFE postgrad
**Duration:** ~2 hours core + open-ended stretch
**Goal:** build a char-level GPT in PyTorch on TinyShakespeare from scratch — no magic.

You will:

1. Tokenize text at the character level.
2. Build a bigram baseline.
3. Implement self-attention by hand.
4. Stack into a multi-head transformer block.
5. Build a small GPT (≈1–3M params) and train it.
6. Sample text. Compare temperatures.

Then, if time, pick from a long list of stretch implementations: RMSNorm, SwiGLU, RoPE,
KV cache, FlashAttention, top-p sampling, attention map visualisation, BPE tokenizer,
checkpointing, and more.

Reference: Andrej Karpathy's *Let's build GPT* lecture / nanoGPT repo.

> Tip: defaults are sized for laptop CPU (~10–15 min training).
> GPU configs are commented at the train cell.


## 0 · Install dependencies

Run this cell **once**. Skips installs already present. `tiktoken` only needed for
the BPE stretch goal (§S8); the rest is required for every section.

If you're on a fresh environment without CUDA, swap the torch line for the CPU build:
```
%pip install torch torchvision torchaudio
```


In [ ]:
%pip install --quiet --upgrade \
    "torch>=2.3" \
    "matplotlib>=3.7" \
    "tiktoken>=0.7" \
    "numpy>=1.26" \
    "datasets>=2.18" \
    "transformers>=4.40" \
    "accelerate>=0.30"
print("deps ready")


## 1 · Setup

In [ ]:
import os, math, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
torch.manual_seed(1337)

DATA_PATH = "tinyshakespeare.txt"
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)
with open(DATA_PATH) as f:
    text = f.read()
print(f"chars: {len(text):,}  preview: {text[:80]!r}")


## 2 · Character-level tokenizer

We treat each unique character in the corpus as a token. Tiny vocab, simple encode/decode.

(Real LLMs use BPE — see stretch goal §S8 at the end.)


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)

print("vocab_size:", vocab_size)
print("encode('hii there'):", encode("hii there"))
print("decode([46,47,47,1,58,46,43,56,43]):", decode([46,47,47,1,58,46,43,56,43]))


## 3 · Train/val split + batching

`block_size` is the maximum context length. `get_batch` randomly samples `batch_size`
contiguous chunks; the target is the next token at every position.


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 128   # context length for laptop. Bump to 256/512 on GPU.
batch_size = 32

def get_batch(split: str):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i : i + block_size] for i in ix])
    y = torch.stack([src[i + 1 : i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("xb.shape:", xb.shape, "yb.shape:", yb.shape)


## 4 · Bigram baseline

Simplest possible LM: each token id directly indexes a `vocab × vocab` table of
next-token logits. No context beyond the previous token. Should hit ≈ 2.4 nats loss.


In [ ]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.tok_emb(idx)  # (B, T, V)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

m = BigramLM(vocab_size).to(device)
print("untrained sample:")
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long, device=device), 200)[0].tolist()))


In [ ]:
opt = torch.optim.AdamW(m.parameters(), lr=1e-2)
for step in range(2000):
    xb, yb = get_batch("train")
    _, loss = m(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"step {step}: loss {loss.item():.4f}")
print(f"final loss: {loss.item():.4f}")
print("trained sample:")
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long, device=device), 200)[0].tolist()))


## 5 · Self-attention from scratch

This is the heart of the transformer. For each token we compute query, key, value
projections, then weight values by query·key similarity (with a causal mask).

```
Attention(Q, K, V) = softmax(Q · Kᵀ / √dₖ) · V
```

The causal mask zeroes out future positions so position `t` only sees positions ≤ t.


In [ ]:
class Head(nn.Module):
    """Single self-attention head."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v


## 6 · Multi-head attention

Run `n_head` heads in parallel on different subspaces of size `n_embd / n_head`,
concatenate, project back to `n_embd`.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


## 7 · FeedForward

Per-token MLP with 4× expansion and GELU activation.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


## 8 · Transformer block (pre-norm)

```
x = x + attn(LN(x))
x = x + ffn(LN(x))
```

Residual connections + LayerNorm — gradient highway and stable training at depth.


In [ ]:
class Block(nn.Module):
    """Transformer block (pre-norm)."""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


## 9 · GPT model

Token embedding + positional embedding → N blocks → final LayerNorm → unembedding.

Weight tying between input and output embeddings is standard practice.


In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd=192, n_head=6, n_layer=6,
                 block_size=128, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


## 10 · Training loop

AdamW + gradient clipping. Tracks train/val loss every `eval_interval`.

**Defaults**: laptop CPU, ~10–15 min for 3000 iterations.

**RTX 5090 stretch**: bump `n_embd=384`, `block_size=256`, `max_iters=10_000`
(commented in cell). Minutes to convergence.


In [ ]:
# laptop-friendly config; bump on GPU
config = dict(n_embd=192, n_head=6, n_layer=6, block_size=block_size, dropout=0.1)

# RTX 5090 stretch:
# config = dict(n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.2)

model = GPT(vocab_size, **config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.2f}M")

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)

@torch.no_grad()
def estimate_loss(eval_iters=50):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

max_iters = 3000
eval_interval = 500

t0 = time.time()
losses_history = {"train": [], "val": [], "step": []}
for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        losses_history["train"].append(losses["train"])
        losses_history["val"].append(losses["val"])
        losses_history["step"].append(it)
        print(f"iter {it:5d}  train {losses['train']:.4f}  val {losses['val']:.4f}  "
              f"elapsed {time.time()-t0:.1f}s")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
print(f"done in {time.time()-t0:.1f}s")


## 11 · Loss curve

Visualise the train/val curves. If val starts diverging from train you've begun overfitting —
either stop earlier or add dropout / shrink the model.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(losses_history["step"], losses_history["train"], label="train")
plt.plot(losses_history["step"], losses_history["val"], label="val")
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("Training curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 12 · Sampling

Compare greedy / temperature / top-k. Notice repetition with low temperature, gibberish at
high temperature.


In [ ]:
ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
print("--- temperature 1.0, top_k=None ---")
print(decode(model.generate(ctx, 500)[0].tolist()))

print("\n--- temperature 0.8, top_k=40 ---")
print(decode(model.generate(ctx, 500, temperature=0.8, top_k=40)[0].tolist()))


---

# Stretch goals

These are full reference implementations. Copy + adapt — no TODOs from here on.
Pick whichever excites you.


## §S1 — RMSNorm (drop-in replacement for LayerNorm)

In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm — drops mean centring, just scales by RMS. Used by Llama family."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: (..., dim)
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)

# Quick sanity: equivalent shape, slightly different stats
x = torch.randn(2, 4, 16)
print("RMSNorm out shape:", RMSNorm(16)(x).shape)


*Try:* substitute `nn.LayerNorm` with `RMSNorm` inside `Block`, retrain, compare loss + speed.

## §S2 — SwiGLU FFN (Llama-style)

In [ ]:
class SwiGLU(nn.Module):
    """SwiGLU FFN — Llama-style. Three linears, ~⅔ × 4 × n_embd hidden."""
    def __init__(self, n_embd, hidden_mult=8/3, dropout=0.0):
        super().__init__()
        # round to multiple of 8 for kernel friendliness
        hidden = int(n_embd * hidden_mult)
        hidden = (hidden + 7) // 8 * 8
        self.w_gate = nn.Linear(n_embd, hidden, bias=False)
        self.w_up   = nn.Linear(n_embd, hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))

print("SwiGLU param count for n_embd=192:",
      sum(p.numel() for p in SwiGLU(192).parameters()))


*Try:* replace `FeedForward` with `SwiGLU` in `Block`. Choose `hidden_mult` so total FFN params ≈ unchanged.

## §S3 — FlashAttention via `torch.nn.functional.scaled_dot_product_attention`

PyTorch 2+ ships an optimised attention kernel that auto-selects FlashAttention on
supported GPUs (Ampere+). Drop-in replacement that's faster and supports longer contexts.


In [ ]:
class FlashSelfAttention(nn.Module):
    """Multi-head self-attention via PyTorch's built-in scaled_dot_product_attention,
    which dispatches to FlashAttention on supported GPUs (Ampere+, Hopper, Blackwell)."""
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)                                    # (B, T, 3C)
        q, k, v = qkv.split(C, dim=-1)
        # reshape to (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        # is_causal=True applies the lower-triangular mask
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

# Quick check vs naive head impl:
x = torch.randn(2, 32, 192, device=device)
fa = FlashSelfAttention(192, 6).to(device)
print("FlashSelfAttention out shape:", fa(x).shape)


*Try:* replace `MultiHeadAttention` with `FlashSelfAttention` in `Block`. Time training with `time.time()` before/after — expect ~2× on long contexts.

## §S4 — Top-p (nucleus) sampling

Adapts to local distribution shape: peaked → small set, flat → larger set.
More robust than top-k.


In [ ]:
@torch.no_grad()
def generate_topp(model, idx, max_new_tokens, temperature=1.0, top_p=0.9):
    """Top-p (nucleus) sampling — keep smallest set whose cumulative prob ≥ p."""
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = probs.sort(dim=-1, descending=True)
        cum = sorted_probs.cumsum(dim=-1)
        # mask anything past the nucleus
        sorted_probs[cum - sorted_probs > top_p] = 0
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        sampled = torch.multinomial(sorted_probs, num_samples=1)
        next_id = sorted_idx.gather(-1, sampled)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
print("--- top-p 0.9 ---")
print(decode(generate_topp(model, ctx, 400, temperature=1.0, top_p=0.9)[0].tolist()))


## §S5 — KV cache (illustrative)

The full version requires threading `use_cache` through every layer; here we just show a
single head with the cache logic. See nanoGPT or the Karpathy minGPT repo for a full
implementation.


In [ ]:
class CachedHead(nn.Module):
    """Single self-attention head with KV cache. Subset reproduction of Head."""
    def __init__(self, n_embd, head_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.cache_k = None
        self.cache_v = None

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None

    def forward(self, x, use_cache=False):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        if use_cache:
            if self.cache_k is None:
                self.cache_k, self.cache_v = k, v
            else:
                self.cache_k = torch.cat([self.cache_k, k], dim=1)
                self.cache_v = torch.cat([self.cache_v, v], dim=1)
            k_use, v_use = self.cache_k, self.cache_v
        else:
            k_use, v_use = k, v

        # causal mask only matters during prefill (T > 1); decode passes T = 1
        wei = q @ k_use.transpose(-2, -1) * k_use.shape[-1] ** -0.5
        if not use_cache:
            mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
            wei = wei.masked_fill(~mask, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v_use

# This is illustrative — wiring it through the whole GPT requires rewriting
# Block.forward to thread `use_cache`. See nanoGPT or Karpathy's miniGPT for full impl.
print("CachedHead module ready (illustrative, not wired into full model).")


## §S6 — Attention map visualisation

Look inside the model: which positions does layer 0, head 0 attend to?
Repeat for other heads / layers — different heads specialise in different things.


In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def attention_map_first_head(model, prompt: str):
    """Compute the first layer first head attention map for a prompt."""
    model.eval()
    ids = torch.tensor([encode(prompt)], device=device)
    block = model.blocks[0]
    head = block.attn.heads[0]
    x = model.tok_emb(ids) + model.pos_emb(torch.arange(ids.size(1), device=device))
    x_ln = block.ln1(x)
    B, T, C = x_ln.shape
    k = head.key(x_ln)
    q = head.query(x_ln)
    wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
    wei = wei.masked_fill(head.tril[:T, :T] == 0, float("-inf"))
    wei = F.softmax(wei, dim=-1)[0]  # (T, T)
    return wei.cpu().numpy(), list(prompt)

prompt = "First Citizen:\n"
wei, toks = attention_map_first_head(model, prompt)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(wei, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks)
ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks)
ax.set_xlabel("attended-to (key)")
ax.set_ylabel("attending (query)")
ax.set_title("Layer 0, head 0 — attention weights")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## §S7 — RoPE (Rotary Positional Embedding)

Replace the additive `pos_emb` with rotary positions. Apply `apply_rope` to Q and K
inside each head.


In [ ]:
def precompute_rope_cache(head_dim, max_seq_len, base=10000.0, device="cpu"):
    """Precompute cos/sin tables for RoPE."""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)        # (max_seq_len, head_dim/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """Rotate pairs of dims of x by (cos, sin) at each position.
    x: (B, n_head, T, head_dim) — head_dim must be even."""
    B, H, T, D = x.shape
    cos = cos[:T].view(1, 1, T, D // 2)
    sin = sin[:T].view(1, 1, T, D // 2)
    x1, x2 = x[..., : D // 2], x[..., D // 2 :]
    rot1 = x1 * cos - x2 * sin
    rot2 = x1 * sin + x2 * cos
    return torch.cat([rot1, rot2], dim=-1)


# Sanity: shape is preserved
cos, sin = precompute_rope_cache(64, 256, device=device)
x = torch.randn(2, 4, 32, 64, device=device)
print("RoPE applied shape:", apply_rope(x, cos, sin).shape)


*Try:* remove `pos_emb`, precompute `cos/sin` once, apply inside each head's `forward` to `q` and `k` before computing attention.

## §S8 — BPE tokenizer (tiktoken)

Replace char tokenizer with a subword tokenizer. Vocab grows to 50k+, but a sentence
shrinks to 5–10× fewer tokens — context window suddenly covers paragraphs.


In [ ]:
# Stretch: drop in tiktoken instead of char-level. Vocab ~50k, but text remains the same
# so lots of common Shakespeare words become single tokens.
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    sample = "First Citizen: Before we proceed any further, hear me speak."
    tokens = enc.encode(sample)
    print(f"tiktoken vocab: {enc.n_vocab}")
    print(f"sample text  : {sample!r}")
    print(f"as {len(tokens)} tokens: {tokens}")
    print(f"decode round-trip: {enc.decode(tokens)!r}")
except ImportError:
    print("pip install tiktoken — then re-run this cell.")


## §S9 — Save / load checkpoint


In [ ]:
# Save trained model state dict and config to disk
ckpt = {"state_dict": model.state_dict(), "config": config, "vocab": chars}
torch.save(ckpt, "gpt-tinyshakespeare.pt")
print("saved gpt-tinyshakespeare.pt")

# Reload
ckpt2 = torch.load("gpt-tinyshakespeare.pt", map_location=device, weights_only=False)
model2 = GPT(vocab_size, **ckpt2["config"]).to(device)
model2.load_state_dict(ckpt2["state_dict"])
model2.eval()
print("reloaded; quick sample:")
print(decode(model2.generate(torch.zeros((1,1), dtype=torch.long, device=device), 200,
                             temperature=0.8, top_k=40)[0].tolist()))


---

# §13 · Ablation comparison — every modern trick, side by side

We now sweep through the modern transformer tricks **one at a time** on the same
TinyShakespeare data, identical model size, identical training budget, and plot
their val-loss curves on a single chart.

Variants:

1. **baseline** — LayerNorm + GELU FFN + naive attention + learned pos
2. **+RMSNorm** (no mean centering)
3. **+SwiGLU FFN** (Llama-style gated activation)
4. **+FlashAttention** (`F.scaled_dot_product_attention`)
5. **+RoPE** (rotary position, no `pos_emb`)
6. **all-modern (Llama-style)** — RMSNorm + SwiGLU + Flash + RoPE
7. **best-config** — all-modern **+** GPT-2 residual-projection init scaling
   (multiply residual proj weights by `1/√(2·n_layer)`) **+** warmup → cosine LR

The §S sections defined the pieces; here we wire them together into one modular
`ModularGPT` so we can flip features with kwargs.


In [ ]:
import time, math
import matplotlib.pyplot as plt

# Ensure all the building blocks from the stretch sections exist
# (RMSNorm, SwiGLU, precompute_rope_cache, apply_rope). Re-define here
# so this section is self-contained — running it does not depend on §S order.

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden_mult=8/3, dropout=0.0):
        super().__init__()
        h = (int(n_embd * hidden_mult) + 7) // 8 * 8
        self.w_gate = nn.Linear(n_embd, h, bias=False)
        self.w_up   = nn.Linear(n_embd, h, bias=False)
        self.w_down = nn.Linear(h, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))

class StandardFFN(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

def _rope_cache(head_dim, max_seq_len, base=10000.0, device="cpu"):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()

def _apply_rope(x, cos, sin):
    # x: (B, n_head, T, head_dim)
    B, H, T, D = x.shape
    cos = cos[:T].view(1, 1, T, D // 2)
    sin = sin[:T].view(1, 1, T, D // 2)
    x1, x2 = x[..., : D // 2], x[..., D // 2 :]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class ModernAttention(nn.Module):
    """Self-attention that can swap in FlashAttention and/or RoPE."""
    def __init__(self, n_embd, n_head, block_size, dropout, use_flash, use_rope):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd)
        self.use_flash = use_flash
        self.use_rope = use_rope
        self.dropout_p = dropout
        self.drop = nn.Dropout(dropout)
        if use_rope:
            cos, sin = _rope_cache(self.head_dim, block_size)
            self.register_buffer("cos", cos)
            self.register_buffer("sin", sin)
        if not use_flash:
            self.register_buffer("tril",
                                 torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if self.use_rope:
            q = _apply_rope(q, self.cos, self.sin)
            k = _apply_rope(k, self.cos, self.sin)
        if self.use_flash:
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True,
                dropout_p=self.dropout_p if self.training else 0.0,
            )
        else:
            scores = q @ k.transpose(-2, -1) * (self.head_dim ** -0.5)
            scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
            scores = F.softmax(scores, dim=-1)
            scores = self.drop(scores)
            out = scores @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class ModernBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout,
                 use_rms, use_swiglu, use_flash, use_rope):
        super().__init__()
        Norm = RMSNorm if use_rms else nn.LayerNorm
        self.norm1 = Norm(n_embd)
        self.attn = ModernAttention(n_embd, n_head, block_size, dropout, use_flash, use_rope)
        self.norm2 = Norm(n_embd)
        self.ff = SwiGLU(n_embd, dropout=dropout) if use_swiglu else StandardFFN(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class ModularGPT(nn.Module):
    def __init__(self, vocab_size, *,
                 n_embd=128, n_head=4, n_layer=3, block_size=128, dropout=0.1,
                 use_rms=False, use_swiglu=False, use_flash=False, use_rope=False,
                 residual_init_scale=False):
        super().__init__()
        Norm = RMSNorm if use_rms else nn.LayerNorm
        self.block_size = block_size
        self.use_rope = use_rope
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = None if use_rope else nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([
            ModernBlock(n_embd, n_head, block_size, dropout,
                        use_rms, use_swiglu, use_flash, use_rope)
            for _ in range(n_layer)
        ])
        self.ln_f = Norm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init)
        if residual_init_scale:
            # GPT-2 trick — scale init of residual projections by 1/sqrt(2*n_layer)
            s = (2 * n_layer) ** -0.5
            for b in self.blocks:
                b.attn.proj.weight.data.mul_(s)
                if use_swiglu:
                    b.ff.w_down.weight.data.mul_(s)
                else:
                    b.ff.net[2].weight.data.mul_(s)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
        return idx


### Training harness

Fixed budget across every variant. The only hyper-parameter difference between
the standard variants and the *best-config* is that best-config uses
`warmup → cosine` learning-rate schedule. Everything else (model size, data,
optimizer betas, weight decay, batch size, iterations) is held constant.


In [ ]:
def cosine_lr(it, max_iters, warmup, peak, min_lr):
    if it < warmup:
        return peak * (it + 1) / warmup
    progress = (it - warmup) / max(1, max_iters - warmup)
    return min_lr + 0.5 * (peak - min_lr) * (1 + math.cos(math.pi * progress))


def train_variant(name, *, get_batch_fn, vocab_size,
                  max_iters=1500, batch_size=64, block_size=128,
                  peak_lr=3e-4, min_lr=3e-5, warmup=150,
                  eval_every=150, eval_iters=20,
                  use_cosine=False, init_model=None, **gpt_kwargs):
    if init_model is None:
        model = ModularGPT(vocab_size, block_size=block_size, **gpt_kwargs).to(device)
    else:
        model = init_model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=peak_lr,
                            betas=(0.9, 0.95), weight_decay=0.1)

    @torch.no_grad()
    def eval_loss():
        model.eval()
        out = {}
        for split in ("train", "val"):
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                xb, yb = get_batch_fn(split)
                _, l = model(xb, yb)
                losses[k] = l.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    steps, val_curve, train_curve = [], [], []
    t0 = time.time()
    for it in range(max_iters):
        if use_cosine:
            for pg in opt.param_groups:
                pg["lr"] = cosine_lr(it, max_iters, warmup, peak_lr, min_lr)
        if it % eval_every == 0 or it == max_iters - 1:
            losses = eval_loss()
            steps.append(it)
            train_curve.append(losses["train"])
            val_curve.append(losses["val"])
        xb, yb = get_batch_fn("train")
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    elapsed = time.time() - t0
    print(f"  {name:48s}  val {val_curve[-1]:.4f}  time {elapsed:5.1f}s  params {n_params/1e6:.2f}M")
    return dict(name=name, model=model, params=n_params,
                steps=steps, train=train_curve, val=val_curve, time=elapsed)


### Run all seven variants

Same TinyShakespeare data, same model dimensions, same 1500 iterations.


In [ ]:
# get_batch already exists from §3 and uses tinyshakespeare. Reuse.

variants = [
    ("baseline (LN + GELU + naive attn + learned pos)", dict(), False),
    ("+RMSNorm",                                       dict(use_rms=True), False),
    ("+SwiGLU",                                        dict(use_swiglu=True), False),
    ("+FlashAttention",                                dict(use_flash=True), False),
    ("+RoPE",                                          dict(use_rope=True), False),
    ("all-modern (Llama-style)",                       dict(use_rms=True, use_swiglu=True,
                                                            use_flash=True, use_rope=True), False),
    ("best-config (all-modern + GPT-2 init + cosine)", dict(use_rms=True, use_swiglu=True,
                                                            use_flash=True, use_rope=True,
                                                            residual_init_scale=True), True),
]

print("training variants on TinyShakespeare (1500 iters each)...\n")
results = {}
for name, flags, use_cosine in variants:
    torch.manual_seed(1337)  # fair start
    results[name] = train_variant(
        name, get_batch_fn=get_batch, vocab_size=vocab_size,
        max_iters=1500, batch_size=64, block_size=block_size,
        use_cosine=use_cosine, **flags,
    )


### Compare — val-loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for name, r in results.items():
    ax.plot(r["steps"], r["val"], label=name, linewidth=1.8 if "best-config" in name else 1.2)
ax.set_xlabel("iteration")
ax.set_ylabel("val loss (cross-entropy)")
ax.set_title("Ablation on TinyShakespeare — modern transformer tricks")
ax.grid(alpha=0.3)
ax.legend(fontsize=8.5, loc="upper right")
plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"{'variant':<55s} {'params (M)':>10s} {'final val':>10s} {'time (s)':>9s}")
print("-" * 86)
for name, r in results.items():
    print(f"{name:<55s} {r['params']/1e6:>10.2f} {r['val'][-1]:>10.4f} {r['time']:>9.1f}")

best_name = min(results, key=lambda k: results[k]["val"][-1])
print(f"\nBest val loss: {best_name}  ->  {results[best_name]['val'][-1]:.4f}")


### Sample from the best model

In [ ]:
best = results[best_name]
ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
print(f"--- sample from: {best_name} ---")
print(decode(best["model"].generate(ctx, 400, temperature=0.8, top_k=40)[0].tolist()))


**What you should see**

- Every modern trick beats the baseline a little. None individually is huge at this scale.
- All-modern (Llama-style) is reliably best of the simple stack.
- The bonus **best-config** layer (GPT-2 residual init + warmup→cosine LR) typically
  improves another ~0.05–0.10 nats of val loss for the same compute.
- At very small scale, gains compress; at frontier scale, the same tricks compound into
  large quality differences (this is why open frontier models all use them).


---

# §14 · Same comparison on a Greek dataset — and continued pretraining

Does the methodology transfer? Do the same tricks help on a different language?
Can we **start from an already-trained model** and continue training on Greek?

### Plan

1. Download a small modern Greek corpus (Greek Wikipedia via 🤗 `datasets`,
   with a Project Gutenberg fallback).
2. Use a **byte-level** tokenizer so the vocabulary (256 bytes) is identical for
   English and Greek — this lets the same model weights transfer.
3. Re-train byte-level on TinyShakespeare so we have an "already trained" English model.
4. Then on Greek, run three experiments with **identical compute budget**:
   - **A** — baseline byte-level GPT, random init, trained on Greek
   - **B** — best-config byte-level GPT, random init, trained on Greek
   - **C** — best-config byte-level GPT, **initialised from the English checkpoint**,
     continued-trained on Greek
5. Plot all three val-loss curves on the same Greek validation set.

`A` vs `B` answers: *do modern tricks help on Greek too?*
`B` vs `C` answers: *does English pretraining transfer to a different alphabet?*


In [ ]:
# Step 1 — fetch Greek text
greek_text = None

def _hf_wiki_greek(budget_chars=1_500_000):
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", "20231101.el",
                      split="train", streaming=True)
    out, total = [], 0
    for item in ds:
        t = item.get("text", "")
        if not t:
            continue
        out.append(t)
        total += len(t)
        if total >= budget_chars:
            break
    return "\n\n".join(out)

def _gutenberg_greek_nt():
    url = "https://www.gutenberg.org/cache/epub/7841/pg7841.txt"
    urllib.request.urlretrieve(url, "greek_nt.txt")
    with open("greek_nt.txt", "r", encoding="utf-8", errors="replace") as f:
        return f.read()

for label, fn in [("HF wikimedia greek wiki", _hf_wiki_greek),
                  ("Project Gutenberg Greek NT", _gutenberg_greek_nt)]:
    try:
        print(f"trying source: {label} ...")
        greek_text = fn()
        print(f"  got {len(greek_text):,} chars")
        break
    except Exception as e:
        print(f"  failed: {type(e).__name__}: {e}")

if greek_text is None:
    raise SystemExit("Could not fetch any Greek source — check internet, "
                     "or `pip install datasets` for the HF fallback.")

print(f"\nGreek corpus: {len(greek_text):,} chars  "
      f"preview: {greek_text[:120]!r}")


In [ ]:
# Step 2 — byte-level tokenizer (vocab = 256, works for any UTF-8 text)
def byte_encode(s: str) -> list[int]:
    return list(s.encode("utf-8"))

def byte_decode(ids: list[int]) -> str:
    return bytes(ids).decode("utf-8", errors="replace")

byte_vocab = 256

# Greek bytes
gbytes = torch.tensor(byte_encode(greek_text), dtype=torch.long)
gn = int(0.9 * len(gbytes))
greek_train, greek_val = gbytes[:gn], gbytes[gn:]
print(f"Greek bytes: {len(gbytes):,}  train {len(greek_train):,}  val {len(greek_val):,}")

# English bytes (re-encode tinyshakespeare as UTF-8 bytes)
ebytes = torch.tensor(byte_encode(text), dtype=torch.long)
en = int(0.9 * len(ebytes))
eng_train_b, eng_val_b = ebytes[:en], ebytes[en:]
print(f"English bytes: {len(ebytes):,}  train {len(eng_train_b):,}  val {len(eng_val_b):,}")

BBSIZE = 128  # block_size for byte experiments

def make_get_batch(train_src, val_src, block_size=BBSIZE, batch_size=64):
    def get_batch(split):
        src = train_src if split == "train" else val_src
        ix = torch.randint(len(src) - block_size - 1, (batch_size,))
        x = torch.stack([src[i : i + block_size] for i in ix])
        y = torch.stack([src[i + 1 : i + 1 + block_size] for i in ix])
        return x.to(device), y.to(device)
    return get_batch

get_batch_eng_b = make_get_batch(eng_train_b, eng_val_b)
get_batch_greek = make_get_batch(greek_train, greek_val)


### Step 3 — pretrain a byte-level English model

This becomes our "already trained model" that we will continue-train on Greek.


In [ ]:
print("pretraining byte-level best-config GPT on TinyShakespeare bytes...\n")
torch.manual_seed(1337)
pretrained_english = train_variant(
    "pretrain on English bytes (best-config)",
    get_batch_fn=get_batch_eng_b, vocab_size=byte_vocab,
    max_iters=2000, batch_size=64, block_size=BBSIZE,
    use_rms=True, use_swiglu=True, use_flash=True, use_rope=True,
    residual_init_scale=True, use_cosine=True,
)


### Step 4 — three Greek runs on identical budget


In [ ]:
print("\ntraining three variants on Greek bytes (1500 iters each)...\n")

torch.manual_seed(1337)
greek_A = train_variant(
    "Greek-A — baseline byte-level GPT (random init)",
    get_batch_fn=get_batch_greek, vocab_size=byte_vocab,
    max_iters=1500, batch_size=64, block_size=BBSIZE,
)

torch.manual_seed(1337)
greek_B = train_variant(
    "Greek-B — best-config byte-level GPT (random init)",
    get_batch_fn=get_batch_greek, vocab_size=byte_vocab,
    max_iters=1500, batch_size=64, block_size=BBSIZE,
    use_rms=True, use_swiglu=True, use_flash=True, use_rope=True,
    residual_init_scale=True, use_cosine=True,
)

# C — start from the English pretrained model and keep training on Greek
import copy
init_C = copy.deepcopy(pretrained_english["model"])
greek_C = train_variant(
    "Greek-C — best-config, init from English checkpoint (continued pretraining)",
    get_batch_fn=get_batch_greek, vocab_size=byte_vocab,
    max_iters=1500, batch_size=64, block_size=BBSIZE,
    init_model=init_C, use_cosine=True,
    peak_lr=1e-4,  # smaller LR for continued pretraining (don't blow up existing weights)
)


### Step 5 — compare on the Greek validation set

In [ ]:
greek_runs = [greek_A, greek_B, greek_C]

fig, ax = plt.subplots(figsize=(10, 5.5))
for r in greek_runs:
    ax.plot(r["steps"], r["val"], label=r["name"], linewidth=1.8)
ax.set_xlabel("iteration")
ax.set_ylabel("val loss on Greek (byte-level CE)")
ax.set_title("Greek experiment — random init vs modern config vs continued-from-English")
ax.grid(alpha=0.3)
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()

print("\nGreek summary:")
print(f"{'run':<70s} {'final val':>10s}  {'time (s)':>9s}")
print("-" * 95)
for r in greek_runs:
    print(f"{r['name']:<70s} {r['val'][-1]:>10.4f}  {r['time']:>9.1f}")

# % improvement of C over A on Greek val loss
imp_BA = (greek_A["val"][-1] - greek_B["val"][-1]) / greek_A["val"][-1] * 100
imp_CA = (greek_A["val"][-1] - greek_C["val"][-1]) / greek_A["val"][-1] * 100
print(f"\nval-loss improvement vs baseline (A):")
print(f"  modern tricks only          (B vs A): {imp_BA:+.2f}%")
print(f"  modern + English pretraining (C vs A): {imp_CA:+.2f}%")


### Greek samples — visualise the difference

In [ ]:
def sample_bytes(model, max_new=300, prompt=""):
    ctx_ids = byte_encode(prompt) if prompt else [byte_encode("\n")[0]]
    ctx = torch.tensor([ctx_ids], dtype=torch.long, device=device)
    out = model.generate(ctx, max_new, temperature=0.8, top_k=40)[0].tolist()
    return byte_decode(out)

for r in greek_runs:
    print(f"\n=== {r['name']} ===")
    print(sample_bytes(r["model"], 240))


**What you should see**

- **B beats A**: even on Greek, modern tricks (RMSNorm + SwiGLU + Flash + RoPE +
  GPT-2 init + cosine LR) reach a lower final val loss in the same iterations.
- **C usually beats B**: starting from the English checkpoint accelerates Greek
  training, even though the *alphabet is different*. Byte-level vocab is shared,
  so the model reuses learned low-level patterns (whitespace, punctuation,
  paragraph rhythm) and only has to specialise the upper layers to Greek.
- The samples from `C` early in training already look more "text-shaped" than `A`,
  even though `C`'s words are not yet meaningful.

### Discussion — why this matters

- The exact same methodology (modular factory + identical budget + plot val
  curves) is how serious research is run. You vary one factor at a time;
  everything else is held fixed.
- Continued pretraining works *across languages* when the tokenisation is shared.
  This is why frontier LLMs use byte-level BPE — same vocab for any UTF-8 text,
  any language, any modality token.
- For Greek-specific work in production: start from a multilingual model
  (Llama 3, Qwen 3, Aya, EuroLLM) and continue-train on a curated Greek corpus
  with a small LR — the same recipe as `Greek-C`, scaled up.

### Further experiments to try

- Hold compute fixed, vary just **one** of {RMSNorm, SwiGLU, Flash, RoPE} on Greek
  and identify which contributes most for Greek specifically.
- Plot the per-byte cross-entropy of a fixed Greek validation set under each model
  on a histogram — see whether the gains are uniform or concentrated in rare bytes.
- Try a different source language (e.g. Latin → Greek) for the pretraining stage,
  and see whether transfer improves vs starting from English.
- Repeat with a real BPE tokenizer (`tiktoken` or train your own SentencePiece on
  the combined English+Greek corpus) and compare bytes/token efficiency.


---

# §15 · Real pretrained model — fine-tune `gpt2` (124M) on Greek

Our byte-level toy in §14 has 0.6M params and trains for seconds. Real practice
starts from a **bigger, already-pretrained** model and continues training.

This section:

1. Loads HuggingFace **`gpt2`** (124M params, English-trained, public weights).
2. Tokenizes the Greek corpus with GPT-2's BPE (Greek encodes as bytes; no vocab
   change required).
3. Measures the **baseline Greek val loss** — how bad is vanilla GPT-2 at Greek?
4. Continues training **two ways** with identical compute:
   - **vanilla recipe** — constant LR, no clipping, default AdamW, no weight decay
   - **modern recipe** — warmup→cosine LR, gradient clipping, AdamW β₂=0.95, weight
     decay 0.1, mixed precision (bf16 on GPU)
5. Plots both curves on the same Greek validation set, prints a summary,
   and generates Greek samples from base / vanilla / modern.

The point: the **methodology** (modular factory, identical compute, paired
recipes, val curve + table + sample) is what scales from the toy 0.6M model up
to a real 124M one. Same code patterns apply at 7B / 70B / 400B if you have the
GPUs.

> **Time on RTX 5090:** ~30s baseline eval + ~1 min per fine-tune = ~3 min total.
> CPU users: bump down to `distilgpt2` (82M) and `max_iters=100`.


In [ ]:
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_MODEL = "gpt2"   # 124M params, English BPE. Greek bytes encode losslessly.
# Alternatives:
#   "distilgpt2"            -> 82M (smaller, faster, less capable)
#   "gpt2-medium"           -> 355M (much slower; ~6× compute per step)
#   "lighteternal/gpt2-greek" -> 124M, pretrained on Greek directly (less improvement
#                                                                   headroom for the demo)
#   "Qwen/Qwen2.5-0.5B"     -> 500M, multilingual; good Greek out of the box

print(f"loading {HF_MODEL} …")
hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)
if hf_tok.pad_token is None:
    hf_tok.pad_token = hf_tok.eos_token

dtype = torch.bfloat16 if device == "cuda" else torch.float32
base_hf = AutoModelForCausalLM.from_pretrained(HF_MODEL, torch_dtype=dtype).to(device)
hf_params = sum(p.numel() for p in base_hf.parameters())
print(f"  loaded {hf_params/1e6:.1f}M parameters on {device} in {dtype}")


In [ ]:
# Tokenize Greek corpus with the HF model's BPE tokenizer
greek_token_ids = hf_tok.encode(greek_text)
gt = torch.tensor(greek_token_ids, dtype=torch.long)
gn = int(0.9 * len(gt))
gtrain_hf, gval_hf = gt[:gn], gt[gn:]
print(f"Greek tokens (GPT-2 BPE): {len(gt):,}")
print(f"  train {len(gtrain_hf):,} tokens, val {len(gval_hf):,} tokens")
print(f"  bytes/token ratio: {len(greek_text.encode('utf-8'))/len(gt):.2f} "
      f"(Greek is much higher than English ~3.6 — BPE was tuned for English)")

HF_BLOCK = 256
HF_BATCH = 8

def hf_get_batch(src, block_size=HF_BLOCK, batch_size=HF_BATCH):
    ix = torch.randint(len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i : i + block_size] for i in ix])
    y = torch.stack([src[i + 1 : i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def hf_eval(model, val_src, iters=10):
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = hf_get_batch(val_src)
        out = model(input_ids=x, labels=y)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / len(losses)

baseline_loss = hf_eval(base_hf, gval_hf, iters=20)
import math as _math
print(f"\nBaseline {HF_MODEL} on Greek (no training): "
      f"val loss = {baseline_loss:.4f}  (perplexity ≈ {_math.exp(baseline_loss):.1f})")


In [ ]:
def finetune_hf(name, *, vanilla, max_iters=400, peak_lr=5e-5):
    """Fine-tune a deepcopy of base_hf on Greek with one of two recipes."""
    model = copy.deepcopy(base_hf).to(device)

    if vanilla:
        opt = torch.optim.AdamW(model.parameters(), lr=peak_lr,
                                betas=(0.9, 0.999), weight_decay=0.0)
        warmup = 0
        min_lr = peak_lr
        do_clip = False
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=peak_lr,
                                betas=(0.9, 0.95), weight_decay=0.1)
        warmup = max(20, max_iters // 20)
        min_lr = peak_lr * 0.1
        do_clip = True

    val0 = hf_eval(model, gval_hf, iters=10)
    steps, curve = [0], [val0]
    print(f"\n=== {name} ===  initial val {val0:.4f}")

    eval_every = max(max_iters // 8, 25)
    t0 = time.time()
    for it in range(max_iters):
        if not vanilla:
            for pg in opt.param_groups:
                pg["lr"] = cosine_lr(it, max_iters, warmup, peak_lr, min_lr)
        x, y = hf_get_batch(gtrain_hf)
        out = model(input_ids=x, labels=y)
        opt.zero_grad(set_to_none=True)
        out.loss.backward()
        if do_clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if (it + 1) % eval_every == 0 or it == max_iters - 1:
            v = hf_eval(model, gval_hf, iters=5)
            steps.append(it + 1)
            curve.append(v)
            print(f"  iter {it+1:4d}: val {v:.4f}  (lr {opt.param_groups[0]['lr']:.2e})")
    elapsed = time.time() - t0
    print(f"  finished in {elapsed:.1f}s -> final val {curve[-1]:.4f}")
    return dict(name=name, model=model, steps=steps, val=curve, time=elapsed)


In [ ]:
HF_ITERS = 400  # bump higher (e.g. 1500) for a stronger demonstration

torch.manual_seed(1337)
hf_vanilla = finetune_hf(
    "vanilla (LR const 5e-5, no clip, β2=0.999, wd=0)",
    vanilla=True, max_iters=HF_ITERS,
)

torch.manual_seed(1337)
hf_modern = finetune_hf(
    "modern (warmup→cosine, grad clip, β2=0.95, wd=0.1)",
    vanilla=False, max_iters=HF_ITERS,
)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axhline(baseline_loss, color="grey", linestyle=":", label=f"baseline {HF_MODEL} (no training)")
ax.plot(hf_vanilla["steps"], hf_vanilla["val"], marker="o", label=hf_vanilla["name"])
ax.plot(hf_modern["steps"],  hf_modern["val"],  marker="s", label=hf_modern["name"])
ax.set_xlabel("iteration")
ax.set_ylabel("Greek val loss")
ax.set_title(f"Fine-tuning {HF_MODEL} ({hf_params/1e6:.0f}M params) on Greek Wiki — vanilla vs modern recipe")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nSummary (Greek val loss):")
print(f"  baseline {HF_MODEL:<40s}  {baseline_loss:.4f}")
print(f"  vanilla recipe                                  {hf_vanilla['val'][-1]:.4f}  "
      f"(Δ {baseline_loss - hf_vanilla['val'][-1]:+.4f})")
print(f"  modern recipe                                   {hf_modern['val'][-1]:.4f}  "
      f"(Δ {baseline_loss - hf_modern['val'][-1]:+.4f})")
gain = (hf_vanilla["val"][-1] - hf_modern["val"][-1]) / hf_vanilla["val"][-1] * 100
print(f"\nmodern recipe beats vanilla by {gain:+.2f}% val loss at identical compute")


In [ ]:
# Generate Greek samples from each model to see qualitative difference
greek_prompt = "Η Ελλάδα είναι"
print(f"prompt: {greek_prompt!r}\n")

prompt_ids = hf_tok.encode(greek_prompt, return_tensors="pt").to(device)
for label, m in [
    (f"{HF_MODEL} (base, no training)", base_hf),
    ("vanilla fine-tune",                hf_vanilla["model"]),
    ("modern fine-tune",                 hf_modern["model"]),
]:
    m.eval()
    with torch.no_grad():
        out = m.generate(prompt_ids, max_new_tokens=100,
                         do_sample=True, temperature=0.8, top_k=40,
                         pad_token_id=hf_tok.eos_token_id)
    print(f"--- {label} ---")
    print(hf_tok.decode(out[0], skip_special_tokens=True))
    print()


**What you should see**

- **Baseline `gpt2`** on Greek has very high val loss (~6–7) — perplexity in the
  hundreds. GPT-2 was trained on English; it knows the Latin alphabet but treats
  Greek as a sea of rare byte tokens.
- **Vanilla fine-tune** drops the loss substantially (typically to ~4.5) — Greek
  is learnable, even from an English checkpoint, because the model knows *how
  language works in general* and BPE keeps the vocab shared.
- **Modern recipe** does ~5–10% better than vanilla at the same compute. Same
  ingredients that helped on the toy model help here too: warmup-then-cosine LR,
  gradient clipping (huge for stability with a non-trivial model), AdamW β₂=0.95
  (faster reaction to gradient changes), weight decay 0.1 (regularises).
- Samples from `modern` are usually more coherent Greek than `vanilla` at the
  same iteration count.

### Take this further

- **Bigger base model.** Swap `HF_MODEL = "Qwen/Qwen2.5-0.5B"` or
  `"Qwen/Qwen2.5-1.5B"` — multilingual, much better Greek out of the box; the
  fine-tune still shows clear improvement.
- **Parameter-efficient fine-tuning.** Wrap with PEFT/LoRA
  (`pip install peft`) — train only ~1% of weights, see how the modern recipe
  still beats vanilla and stays under 5% extra params.
- **Better data.** Replace 🤗 Greek Wikipedia with a curated Greek corpus
  (news, books, code). Same recipe; the modern recipe gap grows on harder data.
- **Mixed precision (`torch.autocast`)** — wrap the forward+loss path; useful
  for the 0.5B / 1.5B models on consumer GPUs.
- **Eval beyond loss.** Add a tiny multiple-choice Greek QA benchmark
  (e.g. translate a few HellaSwag items) — track accuracy alongside loss.
- **DPO / preference tuning.** Once SFT-style continued training works, layer
  on a small preference dataset (e.g. 100 pairs of good/bad Greek responses) to
  align the fine-tune for chat. Same modular harness applies.


## §S10–S20 — More to try

A grab-bag of further extensions, ordered roughly by effort. No reference impl —
go off-script.


1.  **Mixed precision (bf16/fp16)** — wrap forward+loss in `torch.autocast`,
    use `GradScaler` for fp16. Halve memory, near-fp32 quality.
2.  **Cosine LR schedule with warmup** — implement a manual scheduler:
    `lr = (it/warmup) * peak_lr` for `it < warmup`,
    then `lr = min_lr + 0.5*(peak-min)*(1+cos(π*progress))`.
3.  **Grouped-Query Attention (GQA)** — share K, V across groups of heads.
    Compare loss vs MHA at matched param count.
4.  **Multi-Query Attention (MQA)** — extreme form of GQA: 1 KV pair total.
    Even smaller cache, slight quality cost.
5.  **Sliding-window attention** — modify causal mask to also zero out
    positions further than `w` tokens back. Check if loss is still tolerable
    at `w = 64` for `block_size = 256`.
6.  **Tiny MoE block** — replace FFN with 4 expert FFNs + a gating router.
    Pick top-2 experts per token. Watch the auxiliary load-balance loss.
7.  **Curriculum learning** — train first on `block_size = 32`, then 64, then 128.
    Does it converge faster than starting at 128?
8.  **Generate with a system prompt** — concatenate `"ROMEO:\n"` to the
    context tensor before generating. See if the model honours the role.
9.  **Activation function ablation** — train identical models with ReLU, GELU,
    SwiGLU; compare final loss and time to convergence.
10. **Quantization** — after training, quantize weights to int8 with
    `torch.quantization` or `bitsandbytes`. Compare sample quality + memory.
11. **Distillation** — train a tiny student model to match the soft-label
    output of the trained model. Verify the student approaches the teacher's loss.
12. **Speculative decoding** — generate with a small draft model + verify each
    K-step proposal with the big model. Measure tokens/sec.
13. **Mini SFT on instruction data** — assemble a small `(prompt, response)`
    dataset (you can fabricate Shakespearean instructions), fine-tune the
    pretrained model with the same cross-entropy loss masked to the response.
14. **Tokenisation analysis** — for a multilingual sentence, compare byte
    counts under char vs tiktoken vs SentencePiece. Plot tokens-per-language.
15. **Export to ONNX or GGUF** — convert the model so it can be served by
    `onnxruntime` or `llama.cpp` (after writing a small conversion script).

## Resources

- Karpathy — *Let's build GPT, from scratch* (YouTube + nanoGPT repo)
- Vaswani et al. 2017 — *Attention Is All You Need*
- Su et al. 2021 — *RoFormer / RoPE*
- Shazeer 2020 — *GLU Variants Improve Transformer* (SwiGLU)
- Zhang & Sennrich 2019 — *Root Mean Square Layer Normalization* (RMSNorm)
- Tri Dao 2022/2023/2024 — *FlashAttention 1/2/3*
- DeepSeek-V3 + R1 technical reports (2024)
- Rafailov et al. 2023 — *Direct Preference Optimization* (DPO)
- Hoffmann et al. 2022 — *Chinchilla* (scaling laws)
- HuggingFace blog: <https://huggingface.co/blog>
- Lilian Weng's blog: <https://lilianweng.github.io>
